# Preprocessing — Dataset 2 (produksi, noisy)

Sama seperti D1: keluarannya **manifest** + **split** yang dibekukan ke file, bukan fitur.
Bedanya, di D2 langkah *grouping* adalah **nyawa** preprocessing — bukan formalitas.

## Kenapa D2 lebih rawan

Tiap rekaman sirine di D2 punya **dua varian overlay** noise:
`mixed_sound_57.wav` dan `mixed_sound_57_1.wav` berasal dari **rekaman sirine yang sama**.
Hash isi file tidak menangkap ini (noise beda → byte beda). Akibatnya:

> **1675 file hanya berasal dari 1048 rekaman unik.** Dengan split acak, ~65% file test
> punya "saudara" di train → akurasi menggelembung dan tidak bertahan di dunia nyata.

Karena itu split **wajib** group-aware pada `source_id`.

## Yang dikerjakan

| Langkah | D2 |
|---|---|
| Inventaris + `source_id` | dari `mixed_sound_N[_1].wav` dan `sound_N.wav` (traffic) |
| Duplikat / corrupt | tidak ada (EDA) — tetap dicek |
| Fungsi audio bersih | resample 22050 · mono · peak-normalize (divalidasi) |
| Split | 70/15/15, **GROUP-AWARE wajib**, stratified per kelas |

In [1]:
import hashlib
import re
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------ config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
D2_ROOT = ROOT / "Dataset" / "Dataset2"
ML_DIR = ROOT / "ml"
ML_DIR.mkdir(parents=True, exist_ok=True)

# tiap kelas punya lokasi sendiri — traffic terpisah dari sirine (lihat CLAUDE.md)
CLASS_DIRS = {
    "ambulance": D2_ROOT / "Dataset" / "Dataset" / "ambulance",
    "firetruck": D2_ROOT / "Dataset" / "Dataset" / "firetruck",
    "police":    D2_ROOT / "Dataset" / "Dataset" / "police",
    "traffic":   D2_ROOT / "traffic" / "traffic",
}
CLASSES = list(CLASS_DIRS)

# parameter audio — samakan dengan seluruh pipeline
SR = 22050
DURATION = 3.0
N_SAMPLES = int(SR * DURATION)                        # 66150

TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42

for cls, d in CLASS_DIRS.items():
    assert d.exists(), f"folder {cls} tidak ditemukan: {d}"
print("semua folder kelas ditemukan")
print(f"output : {ML_DIR}")

semua folder kelas ditemukan
output : D:\Coding Vscode\Siren Classification\ml


---
## 1 · Inventaris + `source_id`

`source_id` = prefix kelas + angka pertama pada nama file. Inilah kunci anti-bocor:

```
mixed_sound_57.wav    -> ambulance_57  \  satu rekaman,
mixed_sound_57_1.wav  -> ambulance_57  /  dua overlay
sound_401.wav         -> traffic_401      (traffic, tanpa pasangan)
```

`re.search(r"(\d+)")` mengambil angka **pertama**, jadi akhiran `_1` pada varian kedua
tidak membuat source berbeda.

In [2]:
def parse_source_id(stem: str, label: str) -> str:
    """Rekaman sumber = prefix kelas + angka pertama di nama file.

    'mixed_sound_57' dan 'mixed_sound_57_1' -> keduanya 'ambulance_57'
    (angka pertama = 57; akhiran _1 diabaikan). Prefix kelas mencegah bentrok
    penomoran antar kelas.
    """
    match = re.search(r"(\d+)", stem)
    base = match.group(1) if match else stem
    return f"{label}_{base}"


def build_inventory(class_dirs: dict) -> pd.DataFrame:
    """Satu baris per file .wav: identitas + source_id + hash isi."""
    rows = []
    for label, folder in class_dirs.items():
        for f in sorted(folder.glob("*.wav")):
            rows.append({
                "filename": f.name,
                "label": label,
                "source_id": parse_source_id(f.stem, label),
                "path": str(f),
                "md5": hashlib.md5(f.read_bytes()).hexdigest(),
            })
    return pd.DataFrame(rows)


df = build_inventory(CLASS_DIRS)
print(f"total file .wav   : {len(df)}")
print(f"source unik       : {df.source_id.nunique()}")
print(f"file per source   : {len(df) / df.source_id.nunique():.2f}")
print(f"per kelas (file)  : {df.label.value_counts().reindex(CLASSES).to_dict()}")
print(f"per kelas (source): {df.groupby('label').source_id.nunique().reindex(CLASSES).to_dict()}")
df.head()

total file .wav   : 1675
source unik       : 1048
file per source   : 1.60
per kelas (file)  : {'ambulance': 400, 'firetruck': 400, 'police': 454, 'traffic': 421}
per kelas (source): {'ambulance': 200, 'firetruck': 200, 'police': 227, 'traffic': 421}


,filename,label,source_id,path,md5
0,mixed_sound_1.wav,ambulance,ambulance_1,D:\Coding Vscode\Siren Classification\Dataset\...,6f5302ea3a07ca704aeb9e17f67fa3f0
1,mixed_sound_10.wav,ambulance,ambulance_10,D:\Coding Vscode\Siren Classification\Dataset\...,5b12062876f4d298ee0f9a0fbda188d1
2,mixed_sound_100.wav,ambulance,ambulance_100,D:\Coding Vscode\Siren Classification\Dataset\...,f172dd145c771d664238f600e607a435
3,mixed_sound_100_1.wav,ambulance,ambulance_100,D:\Coding Vscode\Siren Classification\Dataset\...,81b989f38734e7231a39b04a087ff6ef
4,mixed_sound_101.wav,ambulance,ambulance_101,D:\Coding Vscode\Siren Classification\Dataset\...,cc3cb9f204bbb24f46e1e58854817b2b


---
## 2 · Cek integritas

EDA menyatakan D2 bersih (tidak ada corrupt maupun duplikat byte). Kita tetap
memverifikasinya di sini — kalau tiba-tiba ada, lebih baik ketahuan sekarang daripada
saat training.

In [3]:
n_dup = int(df.md5.duplicated().sum())
print(f"file duplikat (byte-identik) : {n_dup}")
if n_dup:
    display(df[df.md5.duplicated(keep=False)].sort_values("md5")
              [["filename", "label", "source_id"]])
    df = df[~df.md5.duplicated(keep="first")].reset_index(drop=True)
    print(f"-> dibuang, tersisa {len(df)}")
else:
    print("-> tidak ada yang perlu dibuang.")

file duplikat (byte-identik) : 0
-> tidak ada yang perlu dibuang.


---
## 3 · Fungsi audio bersih (validasi saja)

Identik dengan D1 — supaya D1 dan D2 melewati transformasi yang **persis sama**
(syarat agar perbandingan kedua model adil). Resample 22050, mono, peak-normalize,
panjang tetap `N_SAMPLES`. Hanya divalidasi di sini; file tidak ditulis ulang.

In [4]:
def load_clean_audio(path: str) -> np.ndarray:
    """Audio bersih siap-fitur: 22050 Hz, mono, peak-normalized, panjang tetap."""
    y, _ = librosa.load(path, sr=SR, mono=True)          # 44.1k -> 22.05k + mono
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    y = y[:N_SAMPLES]
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y


print("validasi load_clean_audio:")
for cls in CLASSES:
    p = df[df.label == cls].iloc[0].path
    y = load_clean_audio(p)
    assert y.shape == (N_SAMPLES,), f"panjang salah: {y.shape}"
    assert np.abs(y).max() <= 1.0 + 1e-6, "melebihi rentang [-1, 1]"
    print(f"  {cls:10s} -> shape={y.shape}  peak={np.abs(y).max():.3f}  ok")
print("\nsemua sampel lolos validasi.")

validasi load_clean_audio:


  ambulance  -> shape=(66150,)  peak=1.000  ok
  firetruck  -> shape=(66150,)  peak=1.000  ok
  police     -> shape=(66150,)  peak=1.000  ok
  traffic    -> shape=(66150,)  peak=1.000  ok

semua sampel lolos validasi.


---
## 4 · Split train/val/test (group-aware WAJIB)

Sebelum split, kita tunjukkan **kenapa** group-aware wajib: bandingkan berapa source
yang bocor kalau memakai split acak vs group split. Angka inilah pembenaran seluruh
keputusan ini.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# --- 4a. bukti: berapa yang bocor kalau split acak? ---
idx_tr, idx_te = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED, stratify=df.label)
leak_random = len(set(df.source_id.iloc[idx_tr]) & set(df.source_id.iloc[idx_te]))
n_leak_files = int(df.iloc[idx_te].source_id.isin(set(df.source_id.iloc[idx_tr])).sum())

print("Kalau split ACAK (cara yang salah):")
print(f"  {leak_random} source bocor ke train & test")
print(f"  {n_leak_files}/{len(idx_te)} file test ({n_leak_files/len(idx_te):.0%}) "
      f"punya saudara di train")
print("  -> akurasi test menggelembung. Karena itu kita pakai group split di bawah.")

Kalau split ACAK (cara yang salah):


  164 source bocor ke train & test
  164/252 file test (65%) punya saudara di train
  -> akurasi test menggelembung. Karena itu kita pakai group split di bawah.


In [6]:
# --- 4b. split yang benar: group-aware + stratified ---
def stratified_group_split(df, test_size, val_size, seed):
    """Bagi df jadi train/val/test, group-aware + stratified per label.

    Dua tahap StratifiedGroupKFold: pisahkan test dulu, lalu val dari sisanya.
    """
    def carve(frame, frac):
        n_splits = max(2, round(1 / frac))
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        keep_idx, hold_idx = next(sgkf.split(frame, frame.label, groups=frame.source_id))
        return frame.iloc[keep_idx], frame.iloc[hold_idx]

    rest, test = carve(df, test_size)
    train, val = carve(rest, val_size / (1 - test_size))
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


train_df, val_df, test_df = stratified_group_split(df, TEST_SIZE, VAL_SIZE, SEED)

# --- verifikasi tidak ada source yang bocor antar split ---
s_tr, s_va, s_te = (set(d.source_id) for d in (train_df, val_df, test_df))
assert s_tr.isdisjoint(s_va), "source bocor: train & val"
assert s_tr.isdisjoint(s_te), "source bocor: train & test"
assert s_va.isdisjoint(s_te), "source bocor: val & test"
print("OK — tidak ada source_id yang bocor antar split.\n")

summary = pd.DataFrame({
    split: d.label.value_counts().reindex(CLASSES)
    for split, d in [("train", train_df), ("val", val_df), ("test", test_df)]
})
summary.loc["TOTAL"] = summary.sum()
print(summary)
print(f"\nproporsi file : train {len(train_df)/len(df):.0%} · "
      f"val {len(val_df)/len(df):.0%} · test {len(test_df)/len(df):.0%}")

OK — tidak ada source_id yang bocor antar split.

           train  val  test
label                      
ambulance    286   56    58
firetruck    284   60    56
police       324   64    66
traffic      301   60    60
TOTAL       1195  240   240

proporsi file : train 71% · val 14% · test 14%


---
## 5 · Simpan manifest + split

Format identik dengan D1 (`filename, label, source_id, path` relatif) supaya notebook
training bisa memuat D1 dan D2 dengan kode yang sama.

In [7]:
def relpath(p: str) -> str:
    return str(Path(p).relative_to(ROOT)).replace("\\", "/")


def save(frame, name):
    out = frame.copy()
    out["path"] = out.path.map(relpath)
    out = out[["filename", "label", "source_id", "path"]]
    dest = ML_DIR / name
    out.to_csv(dest, index=False)
    print(f"  {name:28s} {len(out):4d} baris -> {dest}")


print("menyimpan:")
save(df, "manifest_d2.csv")
save(train_df, "split_d2_train.csv")
save(val_df, "split_d2_val.csv")
save(test_df, "split_d2_test.csv")
print("\nselesai — D2 siap untuk notebook training (model produksi).")

menyimpan:
  manifest_d2.csv              1675 baris -> D:\Coding Vscode\Siren Classification\ml\manifest_d2.csv
  split_d2_train.csv           1195 baris -> D:\Coding Vscode\Siren Classification\ml\split_d2_train.csv
  split_d2_val.csv              240 baris -> D:\Coding Vscode\Siren Classification\ml\split_d2_val.csv
  split_d2_test.csv             240 baris -> D:\Coding Vscode\Siren Classification\ml\split_d2_test.csv

selesai — D2 siap untuk notebook training (model produksi).


---

**Selanjutnya:** ekstraksi fitur + training transfer learning. D1 dan D2 kini punya
manifest & split berformat sama, jadi kode training bisa dipakai ulang untuk keduanya
dan hasilnya dibandingkan secara adil.